# importinh libraries

In [40]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import torch.nn as nn
import torch
import torch.onnx
from torch.utils.data import DataLoader ,Dataset
import torch.optim as optim

# preprocessing the data

In [18]:
df = pd.read_csv(os.path.join("data" ,"Titanic-Dataset.csv"))
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [19]:
# removing unnecessary columns
df = df[["Pclass" ,"Sex" ,"Age" ,"SibSp" 
         ,"Parch" ,"Fare" ,"Embarked"  ,"Survived"]]
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Pclass    891 non-null    int64  
 1   Sex       891 non-null    str    
 2   Age       714 non-null    float64
 3   SibSp     891 non-null    int64  
 4   Parch     891 non-null    int64  
 5   Fare      891 non-null    float64
 6   Embarked  889 non-null    str    
 7   Survived  891 non-null    int64  
dtypes: float64(2), int64(4), str(2)
memory usage: 55.8 KB


In [20]:
# drop rows with null values
df.dropna(inplace = True)

# encode categorical data
df["Sex"] = LabelEncoder().fit_transform(df["Sex"])
df["Embarked"] = LabelEncoder().fit_transform(df["Embarked"])

# extract the features and the target
x = df.drop(["Survived"] ,axis = 1).values
y = df["Survived"].values

# splitting the data to training and test
x_train ,x_test ,y_train ,y_test = train_test_split(x ,y ,test_size=0.2)

In [21]:
class TitanicDataset(Dataset):

    def __init__(self ,x ,y):
        super().__init__()
        self.x = torch.tensor(x ,dtype=torch.float32)
        self.y = torch.tensor(y ,dtype=torch.float32)
    
    def __len__(self):
        return len(self.x)
    
    def __getitem__(self, index):
        return self.x[index] ,self.y[index]
    
train_ds = TitanicDataset(x_train ,y_train)
train_loader = DataLoader(train_ds ,batch_size = 16 
                          ,shuffle = True)
test_ds = TitanicDataset(x_test ,y_test)
test_loader = DataLoader(test_ds ,batch_size = 16 
                          ,shuffle = True)

# training the model

In [30]:
class TitanicModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(x_train.shape[1] ,16),
            nn.ReLU(),
            nn.Linear(16 ,8) ,
            nn.ReLU(),
            nn.Linear(8,1),
            nn.Sigmoid()
        )
    
    def forward(self ,x):
        return self.net(x)
    
model = TitanicModel()
criteria = nn.BCELoss()
optimizer = optim.Adam(model.parameters() ,lr = 0.001)

In [31]:
# training loop
epochs = 20
for e in range(epochs):
    epoch_loss = 0
    for x,y in train_loader:
        y = y.unsqueeze(1)
        optimizer.zero_grad()
        output = model(x)
        loss = criteria(output ,y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss
    print(f"epoch {e+1} with loss {epoch_loss/len(train_loader)}")

epoch 1 with loss 0.6361344456672668
epoch 2 with loss 0.6158108711242676
epoch 3 with loss 0.6051855087280273
epoch 4 with loss 0.6011384129524231
epoch 5 with loss 0.6092374324798584
epoch 6 with loss 0.5890138745307922
epoch 7 with loss 0.588678240776062
epoch 8 with loss 0.5872371196746826
epoch 9 with loss 0.5763254165649414
epoch 10 with loss 0.5741937160491943
epoch 11 with loss 0.5725888609886169
epoch 12 with loss 0.5724222660064697
epoch 13 with loss 0.5586565136909485
epoch 14 with loss 0.5615837574005127
epoch 15 with loss 0.5578367114067078
epoch 16 with loss 0.5504770874977112
epoch 17 with loss 0.5573931336402893
epoch 18 with loss 0.5512300133705139
epoch 19 with loss 0.5442355275154114
epoch 20 with loss 0.5379535555839539


In [34]:
# test the model
epoch_loss = 0
for x,y in test_loader:
    with torch.no_grad():
        y = y.unsqueeze(1)
        output = model(x)
        loss = criteria(output ,y)
        epoch_loss += loss
print(f"test loss {epoch_loss/len(test_loader)}")

test loss 0.5298085808753967


In [ ]:
# save the model as onnx file 

dummy_input = torch.randn(1,x_train.shape[1])
model.eval()
torch.onnx.export(
    model ,
    dummy_input ,
    os.path.join("models" ,"model.onnx") ,
    export_params = True ,
    opset_version = 18,
    do_constant_folding = True ,
    input_names = ["input"] ,
    output_names = ["output"],
    dynamic_axes = {'input' : {0 :"batch_size"}
                    ,'output' : {0 :"batch_size"}} 
)

/var/folders/lf/c33www711gv9_vm4d8ht63ww0000gn/T/ipykernel_69363/4132907509.py:3: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0428 17:54:03.635000 69363 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::nms
W0428 17:54:03.636000 69363 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::roi_align
W0428 17:54:03.636000 69363 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::roi_pool


[torch.onnx] Obtain model graph for `TitanicModel([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `TitanicModel([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅


/Users/karimalmamlouk/anaconda3/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 18},
            producer_name='pytorch',
            producer_version='2.11.0',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"input"<FLOAT,[batch_size,7]>
            ),
            outputs=(
                %"output"<FLOAT,[batch_size,1]>
            ),
            initializers=(
                %"net.0.weight"<FLOAT,[16,7]>{TorchTensor(...)},
                %"net.0.bias"<FLOAT,[16]>{TorchTensor(...)},
                %"net.2.weight"<FLOAT,[8,16]>{TorchTensor(...)},
                %"net.2.bias"<FLOAT,[8]>{TorchTensor<FLOAT,[8]>(Parameter containing: tensor([-0.0580, -0.0974,  0.0194, -0.0478,  0.2956, -0.2435,  0.2827,  0.0531], requires_grad=True), name='net.2.bias')},
                %"net.4.weight"<FLOAT,[1,8]>{TorchTensor<FLOAT,[1,8]>(Parameter containing: tensor([[ 0.1637,  0.0407,  0